# Pandas TA Classic ([pandas_ta_classic](https://github.com/xgboosted/pandas-ta-classic)) Strategies for Custom Technical Analysis

## Topics
- What is a Pandas TA Strategy?
    - Builtin Strategies: __AllStrategy__ and __CommonStrategy__
    - Creating Strategies
- Watchlist Class
    - Strategy Management and Execution
    - **NOTE:** The **watchlist** module is independent of Pandas TA Classic. To easily use it, copy it from your local pandas_ta_classic installation directory into your project directory.
- Indicator Composition/Chaining for more Complex Strategies
    - Comprehensive Example: _MACD and RSI Momo with BBANDS and SMAs 50 & 200 and Cumulative Log Returns_

In [ ]:
%matplotlib inline
import datetime as dt

from tqdm import tqdm

import pandas as pd
import pandas_ta_classic as ta

# Optional import for AlphaVantage API
try:
    from alphaVantageAPI.alphavantage import (
        AlphaVantage,
    )  # pip install alphaVantage-api
except ImportError:
    print(
        "[!] alphaVantageAPI not available. Install with: pip install alphaVantage-api"
    )
    AlphaVantage = None

from watchlist import Watchlist  # Is this failing? If so, copy it locally. See above.

print(
    f"\nPandas TA Classic v{ta.version}\nTo install the Latest Version:\n$ pip install pandas-ta-classic\n"
)


# What is a Pandas TA Strategy?
A _Strategy_ is a simple way to name and group your favorite TA indicators. Technically, a _Strategy_ is a simple Data Class to contain list of indicators and their parameters. __Note__: _Strategy_ is experimental and subject to change. Pandas TA comes with two basic Strategies: __AllStrategy__ and __CommonStrategy__.

## Strategy Requirements:
- _name_: Some short memorable string.  _Note_: Case-insensitive "All" is reserved.
- _ta_: A list of dicts containing keyword arguments to identify the indicator and the indicator's arguments

## Optional Requirements:
- _description_: A more detailed description of what the Strategy tries to capture. Default: None
- _created_: At datetime string of when it was created. Default: Automatically generated.

### Things to note:
- A Strategy will __fail__ when consumed by Pandas TA if there is no {"kind": "indicator name"} attribute.

# Builtin Examples

### All
Default Values

In [ ]:
AllStrategy = ta.AllStrategy
print("name =", AllStrategy.name)
print("description =", AllStrategy.description)
print("created =", AllStrategy.created)
print("ta =", AllStrategy.ta)

### Common
Default Values

In [ ]:
CommonStrategy = ta.CommonStrategy
print("name =", CommonStrategy.name)
print("description =", CommonStrategy.description)
print("created =", CommonStrategy.created)
print("ta =", CommonStrategy.ta)

# Creating Strategies
Strategies require a **name** and an array of dicts containing the "kind" of indicator ("sma") and other potential parameters for **ta**.

### Simple Strategy A

In [ ]:
custom_a = ta.Strategy(
    name="A", ta=[{"kind": "sma", "length": 50}, {"kind": "sma", "length": 200}]
)
custom_a

### Simple Strategy B

In [ ]:
custom_b = ta.Strategy(
    name="B",
    ta=[
        {"kind": "ema", "length": 8},
        {"kind": "ema", "length": 21},
        {"kind": "log_return", "cumulative": True},
        {"kind": "rsi"},
        {"kind": "supertrend"},
    ],
)
custom_b

### Bad Strategy. (Misspelled Indicator)

In [ ]:
# Misspelled indicator, will fail later when ran with Pandas TA
custom_run_failure = ta.Strategy(name="Runtime Failure", ta=[{"kind": "percet_return"}])
custom_run_failure

# Strategy Management and Execution with _Watchlist_

### Initialize AlphaVantage Data Source

In [ ]:
if AlphaVantage is not None:
    AV = AlphaVantage(
        api_key="YOUR API KEY",
        premium=False,
        output_size="full",
        clean=True,
        export_path=".",
        export=True,
    )
else:
    AV = None
    print("[!] AlphaVantage not installed. Install with: pip install alphaVantage-api")
AV

### Create Watchlist and set it's 'ds' to AlphaVantage

In [ ]:
data_source = "yahoo"  # Default (change to "av" if AlphaVantage is configured)
# data_source = "av"
watch = Watchlist(["SPY", "IWM"], ds_name=data_source, timed=False)

#### Info about the Watchlist. Note, the default Strategy is "All"

In [ ]:
watch

### Help about Watchlist

In [ ]:
help(Watchlist)

### Default Strategy is "Common"

In [ ]:
# No arguments loads all the tickers and applies the Strategy to each ticker.
# The result can be accessed with Watchlist's 'data' property which returns a
# dictionary keyed by ticker and DataFrames as values
watch.load(verbose=True)

In [ ]:
", ".join([f"{t}: {d.shape}" for t, d in watch.data.items()])

In [ ]:
watch.data["SPY"]

In [ ]:
watch.load("SPY", plot=True, mas=True)

## Easy to swap Strategies and run them

### Running Simple Strategy A

In [ ]:
# Load custom_a into Watchlist and verify
watch.strategy = custom_a
# watch.debug = True
watch.strategy

In [ ]:
watch.load("IWM")

### Running Simple Strategy B

In [ ]:
# Load custom_b into Watchlist and verify
watch.strategy = custom_b
watch.strategy

In [ ]:
watch.load("SPY")

### Running Bad Strategy. (Misspelled indicator)

In [ ]:
# Load custom_run_failure into Watchlist and verify
watch.strategy = custom_run_failure
watch.strategy

In [ ]:
try:
    iwm = watch.load("IWM")
except AttributeError as error:
    print(f"[X] Oops! {error}")

# Indicator Composition/Chaining
- When you need an indicator to depend on the value of a prior indicator
- Utilitze _prefix_ or _suffix_ to help identify unique columns or avoid column name clashes.

### Volume MAs and MA chains

In [ ]:
# Set EMA's and SMA's 'close' to 'volume' to create Volume MAs, prefix 'volume' MAs with 'VOLUME' so easy to identify the column
# Take a price EMA and apply LINREG from EMA's output
volmas_price_ma_chain = [
    {"kind": "ema", "close": "volume", "length": 10, "prefix": "VOLUME"},
    {"kind": "sma", "close": "volume", "length": 20, "prefix": "VOLUME"},
    {"kind": "ema", "length": 5},
    {"kind": "linreg", "close": "EMA_5", "length": 8, "prefix": "EMA_5"},
]
vp_ma_chain_ta = ta.Strategy("Volume MAs and Price MA chain", volmas_price_ma_chain)
vp_ma_chain_ta

In [ ]:
# Update the Watchlist
watch.strategy = vp_ma_chain_ta
watch.strategy.name

In [ ]:
spy = watch.load("SPY")
spy

### MACD BBANDS

In [ ]:
# MACD is the initial indicator that BBANDS depends on.
# Set BBANDS's 'close' to MACD's main signal, in this case 'MACD_12_26_9' and add a prefix (or suffix) so it's easier to identify
macd_bands_ta = [
    {"kind": "macd"},
    {
        "kind": "bbands",
        "close": "MACD_12_26_9",
        "length": 20,
        "ddof": 0,
        "prefix": "MACD",
    },
]
macd_bands_ta = ta.Strategy(
    "MACD BBands", macd_bands_ta, f"BBANDS_{macd_bands_ta[1]['length']} applied to MACD"
)
macd_bands_ta

In [ ]:
# Update the Watchlist
watch.strategy = macd_bands_ta
watch.strategy.name

In [ ]:
spy = watch.load("SPY")
spy

# Comprehensive Strategy

### MACD and RSI Momentum with BBANDS and SMAs and Cumulative Log Returns

In [ ]:
momo_bands_sma_ta = [
    {"kind": "sma", "length": 50},
    {"kind": "sma", "length": 200},
    {"kind": "bbands", "length": 20, "ddof": 0},
    {"kind": "macd"},
    {"kind": "rsi"},
    {"kind": "log_return", "cumulative": True},
    {"kind": "sma", "close": "CUMLOGRET_1", "length": 5, "suffix": "CUMLOGRET"},
]
momo_bands_sma_strategy = ta.Strategy(
    "Momo, Bands and SMAs and Cumulative Log Returns",  # name
    momo_bands_sma_ta,  # ta
    "MACD and RSI Momo with BBANDS and SMAs 50 & 200 and Cumulative Log Returns",  # description
)
momo_bands_sma_strategy

In [ ]:
# Update the Watchlist
watch.strategy = momo_bands_sma_strategy
watch.strategy.name

In [ ]:
spy = watch.load("SPY")
# Apply constants to the DataFrame for indicators
spy.ta.constants(True, [0, 30, 70])
spy.tail()

# Additional Strategy Options

The ```params``` keyword takes a _tuple_ as a shorthand to the parameter arguments in order.
* **Note**: If the indicator arguments change, so will results. Breaking Changes will **always** be posted on the README.

The ```col_numbers``` keyword takes a _tuple_ specifying which column to return if the result is a DataFrame.

In [ ]:
params_ta = [
    {"kind": "ema", "params": (10,)},
    # params sets MACD's keyword arguments: fast=9, slow=19, signal=10
    # and returning the 2nd column: histogram
    {"kind": "macd", "params": (9, 19, 10), "col_numbers": (1,)},
    # Selects the Lower and Upper Bands and renames them LB and UB, ignoring the MB
    {"kind": "bbands", "col_numbers": (0, 2), "col_names": ("LB", "UB")},
    {"kind": "log_return", "params": (5, False)},
]
params_ta_strategy = ta.Strategy(
    "EMA, MACD History, Outter BBands, Log Returns",  # name
    params_ta,  # ta
    "EMA, MACD History, BBands(LB, UB), and Log Returns Strategy",  # description
)
params_ta_strategy

In [ ]:
# Update the Watchlist
watch.strategy = params_ta_strategy
watch.strategy.name

In [ ]:
spy = watch.load("SPY")
spy.tail()

# Disclaimer
* All investments involve risk, and the past performance of a security, industry, sector, market, financial product, trading strategy, or individual’s trading does not guarantee future results or returns. Investors are fully responsible for any investment decisions they make. Such decisions should be based solely on an evaluation of their financial circumstances, investment objectives, risk tolerance, and liquidity needs.

* Any opinions, news, research, analyses, prices, or other information offered is provided as general market commentary, and does not constitute investment advice. I will not accept liability for any loss or damage, including without limitation any loss of profit, which may arise directly or indirectly from use of or reliance on such information.